# N3 solution. Guided activity of Notebook 3, solved

**Predicting Flash Floods in Working Landscapes**, Module 3, Event based evaluation and forecast skill
CUAHSI Water Science Exchange 2026

This short notebook holds the completed activity of Notebook 3 (section 7), with its outputs. It rebuilds only what the activity needs, so it runs on its own in a few seconds. The full notebook with every section executed is `03_did_we_call_the_event_solution.ipynb`.

The task. Which decision threshold gives the best CSI? Find it and report the full set of scores at that threshold.

## 0. Setup

In [1]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/AHWALab/Water-Science-Exchange-Workshop.git"
BRANCH = "module-3-evaluation"

# In Colab the helper files are not here yet. Copy the branch (public, no account needed).
if not Path("hymod_tools.py").exists():
    if not Path("Water-Science-Exchange-Workshop").is_dir():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", "-b", BRANCH, REPO_URL, "Water-Science-Exchange-Workshop"], check=True)
    os.chdir("Water-Science-Exchange-Workshop/notebooks")
sys.path.insert(0, os.getcwd())
print("Working folder:", os.getcwd())

Working folder: /home/claude/m3/notebooks


## 1. What the activity needs from Notebook 3

The data, the threshold, the ensemble probability, the observed events, the two scoring functions of sections 3 and 4 and the list of decision thresholds of section 5.

In [2]:
import numpy as np
import hymod_tools as h

PP, PET, QQ = h.load_leaf_river("LeafCatch.dat")      # 40 years of daily data, mm per day
N_EVAL = 365 * 5                                        # the five year evaluation window
Qobs = QQ[:N_EVAL]
flood_th = h.flood_threshold(QQ, fraction=0.25)
ensemble, _ = h.generate_ensemble(PP, PET, N_EVAL, Nens=100, seed=7)
obs_events = (Qobs >= flood_th).astype(int)             # 1 on observed flood days
prob_exceed = (ensemble >= flood_th).mean(axis=0)       # the forecast probability

def contingency_table(forecast_event, observed_event):
    """Count H, M, F, C for two arrays of 0 and 1."""
    f = np.asarray(forecast_event).astype(bool)
    o = np.asarray(observed_event).astype(bool)
    H = int(np.sum(f & o))        # forecast yes, observed yes
    M = int(np.sum(~f & o))       # forecast no, observed yes, the missed event
    F = int(np.sum(f & ~o))       # forecast yes, observed no, the false alarm
    C = int(np.sum(~f & ~o))      # forecast no, observed no
    return {"H": H, "M": M, "F": F, "C": C}

def categorical_scores(ct):
    """POD, FAR, POFD, CSI, success ratio and frequency bias from one table."""
    H, M, F, C = ct["H"], ct["M"], ct["F"], ct["C"]
    def safe(num, den):
        return num / den if den > 0 else np.nan
    return {"POD": safe(H, H + M),         # hit rate, 1 is best
            "FAR": safe(F, H + F),         # false alarm ratio, 0 is best
            "POFD": safe(F, F + C),        # false alarm rate, 0 is best, ROC x axis
            "CSI": safe(H, H + M + F),     # threat score, 1 is best
            "SR": safe(H, H + F),          # success ratio, 1 minus FAR
            "BIAS": safe(H + F, H + M)}    # frequency bias, 1 means as many forecasts as events

p_thresholds = np.linspace(0, 1, 41)                    # decision thresholds, 0 to 1 in steps of 0.025

## 2. The activity, solved

In [3]:
#################### Your Code Goes Here #########################
# Step 1. CSI at every threshold in p_thresholds (categorical_scores and contingency_table).
csi_values = []
for p_star in p_thresholds:
    s = categorical_scores(contingency_table((prob_exceed >= p_star).astype(int), obs_events))
    csi_values.append(s["CSI"])
csi_values = np.array(csi_values)

# Step 2. The threshold with the largest CSI (np.nanargmax).
best = int(np.nanargmax(csi_values))
p_best = p_thresholds[best]

# Step 3. Print that threshold and its POD, FAR, POFD, CSI and frequency bias.
s = categorical_scores(contingency_table((prob_exceed >= p_best).astype(int), obs_events))
print(f"best CSI at p* = {p_best:.3f}")
for k in ["POD", "FAR", "POFD", "CSI", "BIAS"]:
    print(f"   {k:<5} = {s[k]:.3f}")
###################################################################

best CSI at p* = 0.025
   POD   = 0.443
   FAR   = 0.748
   POFD  = 0.088
   CSI   = 0.192
   BIAS  = 1.757


Reading the result. The best CSI sits at a very low threshold, because the raw ensemble never issues more than about 0.15. At that point the frequency bias is well above 1, the system warns far more often than floods occur. A warning office would weigh the cost of those false alarms, which the CSI alone does not.